<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session12_tool_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۲: فراخوانی ابزار (Tool Calling)

### اهداف این جلسه

- فهم اینکه چرا مدل زبانی به ابزار نیاز دارد و Tool Calling چطور کار می‌کند
- تعریف ابزار با JSON Schema و ساخت خودکار آن از تابع پایتون
- پیاده‌سازی حلقهٔ فراخوانی ابزار، شامل فراخوانی‌های چندگانه و موازی
- اعتبارسنجی آرگومان‌ها و برگرداندن خطا به مدل برای اصلاح
- کنترل انتخاب ابزار با tool_choice و احتیاط دربارهٔ ابزارهایی که داده را تغییر می‌دهند

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. چرا ابزار؟

مدل زبانی سه محدودیت اساسی دارد که با پرامپت و RAG حل نمی‌شوند:

- **دادهٔ زنده:** موجودی انبار، وضعیت سفارش و قیمت امروز در وزن‌های مدل نیستند و در اسناد ثابت RAG هم نیستند.
- **محاسبهٔ دقیق:** مدل حساب را «حدس» می‌زند. ضرب دو عدد هشت‌رقمی برایش قابل‌اعتماد نیست.
- **انجام کار:** مدل فقط متن تولید می‌کند. ثبت سفارش یا ارسال ایمیل کاری است که باید کد انجام دهد.

**فراخوانی ابزار (Tool Calling یا Function Calling)** این شکاف را پر می‌کند. فهرست ابزارها، شامل نام، توضیح و پارامترها، به مدل داده می‌شود. مدل به جای پاسخ متنی می‌تواند بگوید «ابزار X را با این آرگومان‌ها صدا بزن». **کد ما** ابزار را اجرا می‌کند و نتیجه را به مدل برمی‌گرداند. مدل با این نتیجه پاسخ نهایی را می‌سازد.

نکتهٔ کلیدی: **مدل هیچ ابزاری را خودش اجرا نمی‌کند.** فقط درخواست اجرا را به شکل ساختاریافته تولید می‌کند. اجرا، کنترل دسترسی و امنیت کاملاً در دست کد ماست.

مدل `qwen2.5:7b` برای فراخوانی ابزار آموزش دیده و Ollama این قابلیت را در API سازگار با OpenAI پشتیبانی می‌کند.

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. اولین ابزار، دستی

ابزار با یک **JSON Schema** معرفی می‌شود؛ همان قالبی که در جلسهٔ ۴ برای خروجی ساختاریافته دیدیم. توضیح ابزار و پارامترها بخشی از پرامپت است و مدل بر اساس آن تصمیم می‌گیرد کدام ابزار را صدا بزند.

</div>

In [ ]:
import json

PRICES = {"P1001": 28_900_000, "P1003": 21_400_000, "P2001": 41_500_000}


def get_price(product_id):
    return {"product_id": product_id, "price_toman": PRICES.get(product_id)}


price_tool = {
    "type": "function",
    "function": {
        "name": "get_price",
        "description": "Return the current price of a Novin Kala product in toman.",
        "parameters": {
            "type": "object",
            "properties": {"product_id": {"type": "string", "description": "Product id, e.g. P1001"}},
            "required": ["product_id"],
        },
    },
}

messages = [{"role": "user", "content": "قیمت کالای P1003 چنده؟"}]
response = client.chat.completions.create(model=MODEL, messages=messages, tools=[price_tool], temperature=0)
message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:", message.content)
print("tool_calls:", message.tool_calls)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل متنی ننوشته و به جای آن یک `tool_call` با نام ابزار و آرگومان‌ها به شکل رشتهٔ JSON برگردانده است. حالا ابزار را اجرا می‌کنیم و نتیجه را با نقش ویژهٔ `tool` برمی‌گردانیم. شناسهٔ `tool_call_id` نشان می‌دهد این نتیجه پاسخ کدام فراخوانی است.

</div>

In [ ]:
call = message.tool_calls[0]
result = get_price(**json.loads(call.function.arguments))

messages.append(message.model_dump(exclude_none=True))           # the assistant turn with the tool call
messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=[price_tool], temperature=0)
print(final.choices[0].message.content)
for m in messages:
    print(m["role"], "->", m.get("content") or m.get("tool_calls"))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. ساخت خودکار تعریف ابزار

نوشتن JSON Schema با دست تکراری و پرخطاست. در فایل [`store_tools.py`](../store_tools.py) تابع `tool_schema` این کار را از روی **امضای تابع و docstring** انجام می‌دهد: نوع پارامترها از Type Hint، پیش‌فرض‌ها از امضا و توضیح‌ها از بخش `Args` در docstring. Pydantic همان ابزاری است که Schema را می‌سازد. کتابخانه‌هایی مثل LangChain و MCP (جلسهٔ ۱۴) هم همین کار را می‌کنند.

این فایل ابزارهای فروشگاه نوین‌کالا را دارد که در جلسه‌های بعد هم استفاده می‌شوند:

</div>

In [ ]:
import inspect

from store_tools import STORE_TOOLS, place_order, run_tool, search_products, tool_schema

print(inspect.getsource(search_products)[:900])
print(json.dumps(tool_schema(search_products), ensure_ascii=False, indent=2))

In [ ]:
for fn in STORE_TOOLS:
    print(f"{fn.__name__:16} {tool_schema(fn)['function']['description']}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. حلقهٔ فراخوانی ابزار

یک پرسش ممکن است چند ابزار پشت سر هم لازم داشته باشد. مثلاً «ارزان‌ترین لپ‌تاپ را با ارسال سریع به تهران بخرم، جمعاً چقدر می‌شود؟» یعنی جستجوی کالا، هزینهٔ ارسال و محاسبه. مدل‌ها همچنین می‌توانند در یک نوبت **چند فراخوانی موازی** درخواست کنند. حلقهٔ استاندارد:

1. پیام‌ها و ابزارها را به مدل بده.
2. اگر پاسخ `tool_calls` دارد، همه را اجرا کن، نتایج را با نقش `tool` اضافه کن و به گام ۱ برگرد.
3. اگر پاسخ متنی است، تمام شده است.
4. برای جلوگیری از حلقهٔ بی‌پایان، تعداد دورها را محدود کن.

`run_tool` هر فراخوانی را با Pydantic اعتبارسنجی می‌کند و هر خطا را به جای توقف برنامه، به شکل JSON برمی‌گرداند.

</div>

In [ ]:
SYSTEM = ("You are the assistant of Novin Kala, an Iranian online store. Answer in Persian. "
          "Use the tools for prices, stock, shipping and arithmetic; never guess numbers.")


def run_with_tools(question, tools=STORE_TOOLS, max_rounds=6, verbose=True, **kwargs):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    schemas = [tool_schema(fn) for fn in tools]
    for _ in range(max_rounds):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=schemas,
                                                  temperature=0, **kwargs)
        message = response.choices[0].message
        if not message.tool_calls:
            return message.content
        messages.append(message.model_dump(exclude_none=True))
        for call in message.tool_calls:
            result = run_tool(call.function.name, call.function.arguments, tools)
            if verbose:
                print(f"  -> {call.function.name}({call.function.arguments})\n     {result[:200]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "Stopped: too many tool calls."


print(run_with_tools("گوشی A55 سامسونگ موجوده؟ چه رنگ‌هایی داره؟"))

In [ ]:
print(run_with_tools("ارزون‌ترین لپ‌تاپی که دارید چیه؟ اگه دو تا بخرم و با ارسال سریع بفرستید تهران، جمعاً چقدر میشه؟"))

In [ ]:
for q in ["سفارش NK482913 کجاست؟",
          "ارسال سریع به تبریز دارید؟",
          "پایتخت فرانسه کجاست؟"]:
    print(f"\nQ: {q}")
    print("A:", run_with_tools(q))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پرسش آخر ابزاری لازم ندارد و مدل باید مستقیم جواب دهد. انتخاب درست «ابزار نزن» هم بخشی از مهارت مدل است.

## ۵. خطا و اعتبارسنجی

مدل گاهی آرگومان اشتباه می‌فرستد: نوع غلط، شناسهٔ ناموجود یا عبارت محاسباتی نامعتبر. دو قاعده:

1. **هرگز به آرگومان‌های مدل اعتماد نکنید.** آن‌ها را مثل ورودی کاربر اعتبارسنجی کنید. ماشین‌حساب ما همان نسخهٔ امن جلسهٔ ۳ است و `eval` نمی‌کند.
2. **خطا را به مدل برگردانید.** پیام خطای روشن به مدل اجازه می‌دهد خودش را اصلاح کند.

</div>

In [ ]:
print(run_tool("get_product", '{"product_id": "P9999"}', STORE_TOOLS))
print(run_tool("calculator", '{"expression": "__import__(\'os\').getcwd()"}', STORE_TOOLS))
print(run_tool("shipping_cost", '{"order_total": "زیاد", "method": "standard", "city": "آمل"}', STORE_TOOLS))
print(run_tool("delete_database", "{}", STORE_TOOLS))

In [ ]:
print(run_with_tools("مشخصات کالای P9999 رو بده. اگه نبود، ارزون‌ترین تلویزیون رو معرفی کن."))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. کنترل انتخاب ابزار

پارامتر `tool_choice` تعیین می‌کند مدل چطور ابزار انتخاب کند:

| مقدار | رفتار |
|---|---|
| `"auto"` (پیش‌فرض) | مدل خودش تصمیم می‌گیرد ابزار بزند یا مستقیم جواب دهد |
| `"required"` | مدل حتماً باید دست‌کم یک ابزار بزند |
| `"none"` | ابزارها دیده می‌شوند ولی فراخوانی نمی‌شوند |
| `{"type": "function", "function": {"name": "..."}}` | فقط همین ابزار |

پشتیبانی از همهٔ حالت‌ها در سرویس‌ها و مدل‌های مختلف یکسان نیست. اگر رفتار خاصی برایتان حیاتی است، آن را روی همان مدل و سرویس آزمایش کنید. راه مطمئن‌تر برای محدود کردن ابزارها، **فقط فرستادن ابزارهای مجاز** در هر مرحله است.

## ۷. ابزارهایی که داده را تغییر می‌دهند

ابزارهای فقط‌خواندنی، مثل جستجو و محاسبه، کم‌خطرند. ابزاری مثل `place_order` داده را تغییر می‌دهد و هزینهٔ واقعی دارد. اگر مدل اشتباه کند، یا یک متن مخرب در اسناد مدل را فریب دهد (جلسهٔ ۱۴)، سفارش اشتباه ثبت می‌شود. قاعدهٔ عملی: **پیش از اجرای هر ابزار تغییردهنده، از کاربر تأیید صریح بگیرید.**

</div>

In [ ]:
def confirm_then_run(name, arguments, tools):
    if name == "place_order":
        print(f"The assistant wants to run {name}({arguments})")
        if input("Confirm? (y/n) ").strip().lower() != "y":
            return json.dumps({"error": "the customer did not confirm the order"})
    return run_tool(name, arguments, tools)


messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": "یه گلکسی A55 آبی برای شیراز سفارش بده."}]
tools = STORE_TOOLS + [place_order]
for _ in range(6):
    message = client.chat.completions.create(model=MODEL, messages=messages,
                                             tools=[tool_schema(t) for t in tools], temperature=0).choices[0].message
    if not message.tool_calls:
        print(message.content)
        break
    messages.append(message.model_dump(exclude_none=True))
    for call in message.tool_calls:
        messages.append({"role": "tool", "tool_call_id": call.id,
                         "content": confirm_then_run(call.function.name, call.function.arguments, tools)})

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۸. نکته‌های طراحی ابزار

| نکته | دلیل |
|---|---|
| ابزارهای کم و با نام روشن (`search_products` نه `sp`) | ابزارهای زیاد و مشابه، انتخاب را برای مدل سخت می‌کنند |
| توضیح دقیق، شامل قالب ورودی و مثال | توضیح ابزار بخشی از پرامپت است |
| خروجی کوتاه و ساختاریافته | خروجی ابزار وارد پنجرهٔ متن می‌شود و هزینه دارد |
| خطای قابل فهم | مدل بتواند خودش را اصلاح کند |
| کمترین دسترسی لازم | ابزار فقط کاری را بکند که لازم است؛ مثلاً کوئری پارامتری، نه SQL آزاد |

## ۹. جمع‌بندی

- Tool Calling به مدل دسترسی به دادهٔ زنده، محاسبهٔ دقیق و انجام کار می‌دهد. مدل فقط درخواست می‌دهد و کد ما اجرا می‌کند.
- ابزار با JSON Schema تعریف می‌شود و می‌توان آن را خودکار از امضا و docstring تابع ساخت.
- حلقهٔ ابزار تا رسیدن به پاسخ متنی ادامه می‌یابد، فراخوانی‌های موازی را پشتیبانی می‌کند و سقف دور دارد.
- آرگومان‌های مدل را مثل ورودی کاربر اعتبارسنجی کنید و خطا را به مدل برگردانید.
- برای ابزارهای تغییردهنده، تأیید صریح کاربر بگیرید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. ابزاری به نام `cancel_order(order_id)` بنویسید که طبق سیاست نوین‌کالا فقط سفارش‌هایی را لغو کند که هنوز «ارسال شده» نیستند. آن را با تأیید کاربر به حلقه اضافه کنید و با هر دو سفارش نمونهٔ دیتابیس آزمایش کنید.
۲. ده پرسش بنویسید و برای هر کدام ابزار درستی را که باید اول صدا زده شود (یا «هیچ‌کدام») مشخص کنید. درصد انتخاب درست ابزار را اندازه بگیرید.
۳. توضیح‌های پارامترها را از docstringها حذف کنید و تمرین ۲ را تکرار کنید. توضیح ابزارها چقدر اثر دارد؟
۴. خروجی `search_products` را طوری تغییر دهید که اگر بیش از ۵ نتیجه وجود دارد، فقط تعداد نتایج و ۵ مورد اول را برگرداند و به مدل بگوید جستجو را دقیق‌تر کند. چرا محدود کردن خروجی ابزار مهم است؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** ابزاری به نام `cancel_order(order_id)` بنویسید که طبق سیاست نوین‌کالا فقط سفارش‌هایی را لغو کند که هنوز «ارسال شده» نیستند. آن را با تأیید کاربر به حلقه اضافه کنید و با هر دو سفارش نمونهٔ دیتابیس آزمایش کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** ده پرسش بنویسید و برای هر کدام ابزار درستی را که باید اول صدا زده شود (یا «هیچ‌کدام») مشخص کنید. درصد انتخاب درست ابزار را اندازه بگیرید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** توضیح‌های پارامترها را از docstringها حذف کنید و تمرین ۲ را تکرار کنید. توضیح ابزارها چقدر اثر دارد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** خروجی `search_products` را طوری تغییر دهید که اگر بیش از ۵ نتیجه وجود دارد، فقط تعداد نتایج و ۵ مورد اول را برگرداند و به مدل بگوید جستجو را دقیق‌تر کند. چرا محدود کردن خروجی ابزار مهم است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید